# qreg_simulate

> 对应代码：`EconometricsCode/code_in_notes/Chap.4/qreg_simulate.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebooks/PyStata.md`。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.4`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。

In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.4")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

我们构造一个系数随分位点变化的模拟 DGP：x 为自由度 3 的卡方变量加 2（保证为正），q 为均匀分布于 (0,1) 的分位点标记。

In [ ]:
%%stata
clear
set more off
set seed 505
set obs 500
// 生成x
gen x=rchi2(3)+2
// 分位数
gen q=runiform()

关键设定是回归系数依赖于 q：截距 b0=50q²、一次项系数 b1=5+10q、二次项系数 b2=exp(q) 都随分位点上升而增大。于是同一个 x 在不同分位点上对应不同的 y，条件分位数函数是 x 的二次函数且斜率随分位点递增——这是典型的"系数随分位点变化"（heterogeneous effects）数据生成过程。

In [ ]:
%%stata
// 系数
gen b0=50*q^2
gen b1=5+q*10
gen b2=exp(q)
// y
gen y=b0+b1*x+b2*x^2

把 q 固定为 0.25、0.5、0.75，我们得到总体的条件分位数函数 Q_y(τ|x) 在这三个分位点上的解析表达式，用作分位数回归要逼近的真值。

In [ ]:
%%stata
// 条件分位数函数
gen q25 = 50*0.25^2+(5+0.25*10)*x+exp(0.25)*x^2
gen q50 = 50*0.5^2+(5+0.5*10)*x+exp(0.5)*x^2
gen q75 = 50*0.75^2+(5+0.75*10)*x+exp(0.75)*x^2

按 x 排序后叠加散点与三条真实条件分位数曲线：三条曲线之间距离随 x 增大而拉开，直观展示了该 DGP 下条件分布的色散随 x 增大——这正是分位数回归能刻画而 OLS 无法刻画的异质性。

In [ ]:
%%stata
// 画图
sort x
twoway (scatter y x) (line q25 x) (line q50 x) (line q75 x) 